In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ==========================================
# 1. DATA PREPARATION & FEATURE ENGINEERING
# ==========================================
def load_and_prep_data(filepath, sample_size=5000):
    df = pd.read_csv(filepath).dropna().reset_index(drop=True)

    if len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

    bin_cfg = {
        "age":        ([0, 24, 35, 50, 150],                  [0, 1, 2, 3]),
        "monthly_inc":([-1, 2999, 5000, 8000, float("inf")],  [0, 1, 2, 3]),
        "rev_util":   ([-1, .299, .699, .999, float("inf")],  [0, 1, 2, 3]),
        "debt_ratio": ([-1, .199, .349, .499, float("inf")],  [0, 1, 2, 3]),
        "open_credit":([-1, 2, 8, float("inf")],              [0, 1, 2]),
    }
    for col, (bins, labels) in bin_cfg.items():
        df[col] = pd.cut(df[col], bins=bins, labels=labels).astype(float)

    return df.drop("dlq_2yrs", axis=1), df["dlq_2yrs"]


# ==========================================
# 2. MISSING DATA INJECTION (MCAR)
# ==========================================
def inject_mcar(X, rate, seed=42):
    X_miss = X.copy()
    rng = np.random.default_rng(seed)          # modern RNG — faster than np.random.seed
    mask = rng.random(X_miss.shape) < rate
    X_miss[mask] = np.nan
    return X_miss, mask


# ==========================================
# 3. IMPUTATION METHODS
# ==========================================
def build_imputers():
    """
    Returns a dict of (name → fitted-ready imputer).
    Defined once and reused across rates to avoid re-instantiation overhead.
    n_jobs=-1 added wherever the estimator supports parallelism.
    """
    return {
        "Mode": SimpleImputer(strategy="most_frequent"),

        "KNN":  KNNImputer(n_neighbors=5),          # sklearn ≥1.3 supports n_jobs here
                                                     # omitted for compatibility

        "SHD":  None,   # handled separately (not sklearn-compatible)

        "MissForest": IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=10, random_state=42, n_jobs=-1   # ← parallel trees
            ),
            max_iter=5, random_state=42
        ),

        "MICE": IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=10, random_state=42
        ),
    }


def _shd_impute(X_miss):
    """Sequential Hot-Deck: sort by a non-null column, ffill then bfill."""
    # Pick the sort column as the one with fewest missing values (robust fallback)
    sort_col = X_miss.isna().sum().idxmin()
    df = X_miss.sort_values(by=sort_col, na_position="last")
    return df.ffill().bfill().sort_index()


def apply_imputations(X_missing, imputers):
    """
    Fits and transforms X_missing with each imputer.
    Reuses pre-built imputer objects (passed in) to avoid repeated __init__ overhead.
    Returns dict of {name: imputed DataFrame}.
    """
    cols = X_missing.columns
    results = {}

    for name, imp in imputers.items():
        if name == "SHD":
            results[name] = _shd_impute(X_missing)
        else:
            # clone=False: reuse object but re-fit on new data each call
            results[name] = pd.DataFrame(
                imp.fit_transform(X_missing), columns=cols
            )

    return results


# ==========================================
# 4. EVALUATION
# ==========================================
def evaluate_precision(X_true, imputed, mask):
    """Imputation accuracy on the masked ground-truth values."""
    true_flat = X_true.values[mask]
    results = {}
    for name, X_imp in imputed.items():
        pred_flat = X_imp.values[mask]
        valid = ~np.isnan(true_flat)
        results[name] = accuracy_score(
            np.round(true_flat[valid]), np.round(pred_flat[valid])
        )
    return results


def evaluate_classifiers(imputed_train, y_train, X_test_clean, y_test):
    """
    Trains LR + RF on each imputed training set; tests on clean hold-out.
    RF uses n_jobs=-1 for parallel tree building.
    """
    classifiers = {
        "Logistic Regression": LogisticRegression(max_iter=500),
        "Random Forest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1   # ← parallel trees
        ),
    }
    rows = []
    for imp_name, X_tr in imputed_train.items():
        for clf_name, clf in classifiers.items():
            clf.fit(X_tr, y_train)
            y_pred = clf.predict(X_test_clean)
            rows.append({
                "Imputation": imp_name,
                "Classifier":  clf_name,
                "Accuracy":    accuracy_score(y_test, y_pred),
                "Precision":   precision_score(y_test, y_pred, zero_division=0),
                "Recall":      recall_score(y_test, y_pred, zero_division=0),
                "F1_Score":    f1_score(y_test, y_pred, zero_division=0),
            })
    return rows


# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    filepath = "../data/Credit Risk Benchmark Dataset.csv"

    print("Loading and preparing data...")
    X_full, y_full = load_and_prep_data(filepath, sample_size=5000)

    # ── Split ONCE outside the loop (same seed → identical split every time) ──
    X_train_clean, X_test_clean, y_train, y_test = train_test_split(
        X_full, y_full, test_size=0.30, random_state=42
    )

    missing_rates   = [0.10, 0.20, 0.30, 0.40, 0.50]
    precision_report = pd.DataFrame(index=missing_rates)
    classifier_report = []

    for rate in missing_rates:
        print(f"\n--- {int(rate*100)}% Missing ---")

        # Build fresh imputers each rate (must re-fit on new missingness pattern)
        imputers = build_imputers()

        # ── Phase 1: Imputation Precision (full dataset) ──────────────────────
        X_miss_full, mask_full = inject_mcar(X_full, rate)
        imputed_full = apply_imputations(X_miss_full, imputers)
        for name, score in evaluate_precision(X_full, imputed_full, mask_full).items():
            precision_report.loc[rate, name] = score

        # ── Phase 2: Downstream Classifier (train split only) ─────────────────
        # Re-fit imputers on training missingness (avoid leaking test distribution)
        imputers_train = build_imputers()
        X_train_miss, _ = inject_mcar(X_train_clean, rate)
        imputed_train = apply_imputations(X_train_miss, imputers_train)

        for row in evaluate_classifiers(imputed_train, y_train, X_test_clean, y_test):
            row["Missing_Rate"] = rate
            classifier_report.append(row)

    # ── Results ──────────────────────────────────────────────────────────────
    print("\n=== Imputation Precision Score ===")
    print((precision_report.round(4) * 100).to_string())

    df_metrics = pd.DataFrame(classifier_report)
    f1_pivot = df_metrics.pivot_table(
        index=["Missing_Rate", "Classifier"], columns="Imputation", values="F1_Score"
    )
    print("\n=== F1-Scores ===")
    print(f1_pivot.round(4).to_string())

    # ── Export ────────────────────────────────────────────────────────────────
    precision_report.to_csv("Phase1_Precision_Results.csv")
    df_metrics.to_csv("Phase2_Classifier_Metrics_Results.csv", index=False)
    f1_pivot.to_csv("Phase2_F1_Pivot_Table.csv")

    # ── Plot 1: Precision ─────────────────────────────────────────────────────
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=precision_report, markers=True, dashes=False,
                 linewidth=2.5, markersize=8)
    plt.title("Imputation Precision across Missing Data Proportions",
              fontsize=14, fontweight="bold")
    plt.xlabel("Missing Data Proportion", fontsize=12)
    plt.ylabel("Precision Score", fontsize=12)
    plt.xticks(missing_rates, [f"{int(r*100)}%" for r in missing_rates])
    plt.legend(title="Imputation Method", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("Phase1_Precision_Chart.png", dpi=300)
    plt.close()

    # ── Plot 2: RF F1-Score ───────────────────────────────────────────────────
    rf_f1 = df_metrics[df_metrics["Classifier"] == "Random Forest"].pivot_table(
        index="Missing_Rate", columns="Imputation", values="F1_Score"
    )
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=rf_f1, markers=True, dashes=False,
                 linewidth=2.5, markersize=8)
    plt.title("Random Forest F1-Score across Missing Data Proportions",
              fontsize=14, fontweight="bold")
    plt.xlabel("Missing Data Proportion", fontsize=12)
    plt.ylabel("F1-Score", fontsize=12)
    plt.xticks(missing_rates, [f"{int(r*100)}%" for r in missing_rates])
    plt.legend(title="Imputation Method", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("Phase2_RF_F1_Chart.png", dpi=300)
    plt.close()

    print("\nDone. CSVs and charts saved.")

Loading and preparing data...

--- 10% Missing ---

--- 20% Missing ---

--- 30% Missing ---

--- 40% Missing ---

--- 50% Missing ---

=== Imputation Precision Score ===
      Mode    KNN    SHD  MissForest   MICE
0.1  51.91  51.55  42.62       52.54  49.52
0.2  52.15  47.52  43.10       51.20  48.87
0.3  52.12  45.02  37.68       50.12  42.92
0.4  52.24  43.78  36.84       48.93  43.32
0.5  52.01  42.64  42.59       47.19  41.40

=== F1-Scores ===
Imputation                           KNN    MICE  MissForest    Mode     SHD
Missing_Rate Classifier                                                     
0.1          Logistic Regression  0.7484  0.7520      0.7529  0.7620  0.3767
             Random Forest        0.7413  0.7449      0.7435  0.7328  0.4544
0.2          Logistic Regression  0.7338  0.7498      0.7438  0.7656  0.4837
             Random Forest        0.7389  0.7507      0.7337  0.7355  0.4872
0.3          Logistic Regression  0.7368  0.7395      0.7286  0.7688  0.4381
       